# Wind-driven snow effect on projected surface melt

Produces 4 spatial maps for 4 models.
Requires MAR simulations run with (DS) and without (nDS) wind-driven snow.



$$\mathrm{DS}_\% = 100\;\frac{\Delta \mathrm{ME}_\mathrm{DS} - \Delta \mathrm{ME}_\mathrm{nDS}}
{\left|\overline{\mathrm{ME}}^{\,2071-2100}_\mathrm{nDS}\right|},$$

where $\Delta$ is the 2071-2100 minus 1981-2010 difference. 
Dots mark cells where the change in the annual DS-nDS difference is significant at the 5%.

**Input data**

* `MARcst-AN35km-176x148.cdf2` - MAR grid and masks (AIS, ICE, GROUND, SH).
* `year-MAR-{DS,nDS}_{forcing}-1980-2100.nc` - annual MAR output, variable `ME`
  in mm w.e. yr$^{-1}$, on the 35 km Antarctic grid.
* IMBIE-2 drainage basins, `ANT_Basins_IMBIE2_v1.6.shp`, from
  <http://imbie.org/imbie-2016/drainage-basins/>. used for the drainage basins outlines;
  set `SHP_PATH = None` to skip it.

**Dependencies**: numpy, scipy, xarray, netCDF4, matplotlib, geopandas.


In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.stats import f as f_dist, t as t_dist

## Settings

In [ ]:
DATA_DIR = "."
MASK_FILE = f"{DATA_DIR}/MARcst-AN35km-176x148.cdf2"
DS_FILE = DATA_DIR + "/year-MAR-DS_{f}-1980-2100.nc"
NDS_FILE = DATA_DIR + "/year-MAR-nDS_{f}-1980-2100.nc"
SHP_PATH = f"{DATA_DIR}/ANT_Basins_IMBIE2_v1.6.shp"   # None to skip the outlines
FIG_OUT = "ME_DS_effect"

FORCINGS = {"UKM": "MAR-UKESM", "CNRM": "MAR-CNRM",
            "MPI": "MAR-MPI", "IPL": "MAR-IPSL"}

REF_PERIOD = (1981, 2010)
FUT_PERIOD = (2071, 2100)
ALPHA = 0.05

# Normalise by the end-of-century nDS melt (2071-2100). Cells where that reference is weaker

NORMALISE = True
MIN_REF = 5.0        # kg m-2 yr-1
VMAX = None          # colour limit; 

## Grid and masks

`X` and `Y` are the MAR polar-stereographic coordinates in km, on the same
projection as EPSG:3031, which is what lets the IMBIE-2 polygons be overlaid after
a metres-to-km rescaling.

In [ ]:
cst = xr.open_dataset(MASK_FILE, decode_times=False)
x_km = cst["X"].values.astype(float)
y_km = cst["Y"].values.astype(float)
XX, YY = np.meshgrid(x_km, y_km)
NY, NX = XX.shape

ais = cst["AIS"].values.squeeze() > 0
ice = ais & (cst["ICE"].values.squeeze() > 30)
grounded = ice & (cst["GROUND"].values.squeeze() > 30)

print(f"grid {NY} x {NX}, {ice.sum()} ice cells")

### Melt

In [ ]:
def read_melt(path):
    """Annual melt as (time, y, x) in kg m-2 yr-1, with a year coordinate."""
    ds = xr.open_dataset(path, decode_times=False)
    if "SECTOR" in ds.dims:
        ds = ds.isel(SECTOR=0)
    me = ds["ME"].squeeze(drop=True).transpose("TIME", "Y", "X")
    years = np.asarray(ds["TIME"].values, float)
    if not 1500 < np.nanmin(years) < 2500:          # TIME stored as an index
        years = 1980 + np.arange(me.sizes["TIME"])
    return me.values, years.astype(int)


def window(cube, years, period):
    """The years of `cube` falling inside `period`, inclusive."""
    lo, hi = period
    sel = (years >= lo) & (years <= hi)
    if not sel.any():
        raise ValueError(f"years {lo}-{hi} absent, file covers {years.min()}-{years.max()}")
    return cube[sel]

## Significance

What is on spatial maps is the change in $d(t) = \mathrm{ME}_\mathrm{DS}(t) -
\mathrm{ME}_\mathrm{nDS}(t)$. The two 30-year samples of $d$ are compared with a
$t$-test, using Student's version where an $F$-test retains equal variances at the
5% level and Welch's version otherwise.

In [ ]:
def welch_or_student_p(a, b):
    """Two-sided p-value per grid cell for two (time, y, x) samples."""
    na, nb = a.shape[0], b.shape[0]
    va, vb = a.var(axis=0, ddof=1), b.var(axis=0, ddof=1)
    diff = a.mean(axis=0) - b.mean(axis=0)

    with np.errstate(divide="ignore", invalid="ignore"):
        cdf = f_dist.cdf(va / vb, na - 1, nb - 1)
        equal_var = 2 * np.minimum(cdf, 1 - cdf) > 0.05

        pooled = ((na - 1) * va + (nb - 1) * vb) / (na + nb - 2)
        p_student = 2 * t_dist.sf(np.abs(diff / np.sqrt(pooled * (1 / na + 1 / nb))),
                                  na + nb - 2)

        sa, sb = va / na, vb / nb
        dof = (sa + sb) ** 2 / (sa ** 2 / (na - 1) + sb ** 2 / (nb - 1))
        p_welch = 2 * t_dist.sf(np.abs(diff / np.sqrt(sa + sb)), dof)

        p = np.where(equal_var, p_student, p_welch)
    return np.where((va > 0) & (vb > 0), p, np.nan)

In [ ]:
effect, significant, reference = {}, {}, {}

for key in FORCINGS:
    me_ds, years = read_melt(DS_FILE.format(f=key))
    me_nds, _ = read_melt(NDS_FILE.format(f=key))

    change_ds = window(me_ds, years, FUT_PERIOD).mean(0) - window(me_ds, years, REF_PERIOD).mean(0)
    change_nds = window(me_nds, years, FUT_PERIOD).mean(0) - window(me_nds, years, REF_PERIOD).mean(0)
    effect[key] = np.where(ice, change_ds - change_nds, np.nan)
    reference[key] = window(me_nds, years, FUT_PERIOD).mean(0)

    d = me_ds - me_nds
    p = welch_or_student_p(window(d, years, REF_PERIOD), window(d, years, FUT_PERIOD))
    significant[key] = ice & np.isfinite(p) & (p < ALPHA)

    #print(f"{FORCINGS[key]:<11s} median effect {np.nanmedian(effect[key][ice]):6.1f} kg m-2 yr-1, "
          #f"significant on {100 * significant[key][ice].mean():.0f}% of the ice sheet")

In [ ]:
field, blank = {}, {}

for key in FORCINGS:
    if NORMALISE:
        ref = np.abs(reference[key])
        usable = ice & (ref >= MIN_REF)
        field[key] = np.where(usable, 100 * effect[key] / ref, np.nan)
        blank[key] = ice & ~usable
    else:
        field[key] = effect[key]
        blank[key] = np.zeros_like(ice)

pooled = np.concatenate([f[np.isfinite(f)] for f in field.values()])
vmax = VMAX or np.percentile(np.abs(pooled), 98)
vmax = 5 * np.ceil(vmax / 5) if vmax < 50 else 10 * np.ceil(vmax / 10)

label = (f"DS effect on melt  (% of nDS melt in {FUT_PERIOD[0]}-{FUT_PERIOD[1]})"
         if NORMALISE else r"$\Delta$ME, DS $-$ nDS  (kg m$^{-2}$ yr$^{-1}$)")

if NORMALISE:
    dropped = 100 * np.mean([blank[k][ice].mean() for k in FORCINGS])
    print(f"reference melt below {MIN_REF:g} kg m-2 yr-1 on {dropped:.0f}% of the ice sheet")
print(f"colour limit {vmax:.0f}")

### Basin outlines

IMBIE-2 polygons are reprojected to EPSG:3031 and scaled from metres to the
kilometres used by the MAR grid.

In [ ]:
basin_outlines = []

if SHP_PATH:
    import geopandas as gpd

    basins = gpd.read_file(SHP_PATH).to_crs("EPSG:3031")
    basins = basins[basins["Subregion"].str.strip() != "Islands"]
    basins = basins.dissolve(by="Subregion").reset_index()
    geoms = basins.geometry.scale(1e-3, 1e-3, origin=(0, 0)).simplify(2.0)

    for geom in geoms:
        parts = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
        for part in parts:
            for ring in [part.exterior, *part.interiors]:
                basin_outlines.append(np.asarray(ring.coords))

    print(f"{len(basins)} basins, {len(basin_outlines)} outline segments")

## Figure

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "mathtext.fontset": "dejavusans",
})

fig, axes = plt.subplots(2, 2, figsize=(13, 11))

for panel, (ax, key) in enumerate(zip(axes.ravel(), FORCINGS)):
    if blank[key].any():
        ax.pcolormesh(XX, YY, np.where(blank[key], 1.0, np.nan), shading="auto",
                      cmap=ListedColormap(["0.85"]), vmin=0, vmax=1, zorder=2)

    mesh = ax.pcolormesh(XX, YY, field[key], shading="auto", cmap="RdBu_r",
                         vmin=-vmax, vmax=vmax, zorder=3, rasterized=True)

    ax.contour(XX, YY, ice.astype(float), [0.5], colors="0.1", linewidths=0.9, zorder=7)
    ax.contour(XX, YY, grounded.astype(float), [0.5], colors="0.1", linewidths=0.5, zorder=7)
    for outline in basin_outlines:
        ax.plot(outline[:, 0], outline[:, 1], color="0.25", lw=0.45, alpha=0.9, zorder=8)

    dots = significant[key] & np.isfinite(field[key])
    iy, ix = np.where(dots)
    ax.scatter(XX[iy, ix], YY[iy, ix], s=2, c="black", marker=".", lw=0, zorder=10)

    ax.set_title(f"{chr(97 + panel)}  {FORCINGS[key]}", fontsize=18, loc="left")
    ax.set_aspect("equal")
    ax.axis("off")

cax = fig.add_axes([0.90, 0.22, 0.02, 0.55])
cbar = fig.colorbar(mesh, cax=cax, extend="both")
cbar.set_label(label, fontsize=14)
cbar.ax.tick_params(labelsize=14)

fig.patch.set_facecolor("white")
fig.subplots_adjust(wspace=0.06, hspace=0.12, left=0.02, right=0.88, top=0.95, bottom=0.04)
#fig.savefig(f"{FIG_OUT}.png", dpi=300, bbox_inches="tight")
#fig.savefig(f"{FIG_OUT}.pdf", bbox_inches="tight")
plt.show()